# Lab 06.4: Disaggregated Serving

Explore the disaggregated prefill/decode architecture: pool sizing, KV cache transfer
costs across interconnects (RDMA, TCP, NVLink), Mooncake's prediction-based scheduling,
and throughput comparison vs aggregated serving.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
from content.utils.benchmark import Timer
from content.utils.latency import latency_breakdown

## 1. Disaggregated Architecture Overview

In disaggregated serving, **prefill** (compute-bound) and **decode** (memory-bound) phases
run on separate GPU pools. This allows independent scaling and hardware specialization:
- Prefill pool: high-FLOPS GPUs, batch many prompts
- Decode pool: high-bandwidth GPUs, optimized for sequential token generation

The key tradeoff: KV cache must be transferred between pools after prefill completes.

In [ ]:
# Model parameters for KV cache sizing
NUM_LAYERS = 32
NUM_KV_HEADS = 8
HEAD_DIM = 128
DTYPE_BYTES = 2  # FP16

def kv_cache_size_bytes(seq_len, batch_size=1):
    """KV cache size for one request: 2 (K+V) * layers * heads * head_dim * seq_len * dtype."""
    return 2 * NUM_LAYERS * NUM_KV_HEADS * HEAD_DIM * seq_len * DTYPE_BYTES * batch_size

seq_lens = [512, 1024, 2048, 4096, 8192, 16384, 32768]
sizes_mb = [kv_cache_size_bytes(s) / 1e6 for s in seq_lens]

plt.figure(figsize=(8, 4))
plt.bar([str(s) for s in seq_lens], sizes_mb, color='#2563eb')
plt.xlabel('Sequence Length')
plt.ylabel('KV Cache Size (MB)')
plt.title('Per-Request KV Cache Size (32L, 8 KV heads, d=128, FP16)')
plt.tight_layout()
plt.show()
print(f"32K seq -> {sizes_mb[-1]:.1f} MB per request")

## 2. KV Transfer Time: RDMA vs TCP vs NVLink

The critical bottleneck in disaggregated serving is transferring KV cache from prefill to decode nodes.
Transfer time directly impacts time-to-first-token (TTFT) for the decode phase.

In [ ]:
# Interconnect bandwidths (GB/s, effective after protocol overhead)
INTERCONNECTS = {
    'TCP (100GbE)': 10.0,       # ~80 Gbps effective
    'RDMA (200Gb IB)': 22.0,    # ~176 Gbps effective
    'RDMA (400Gb IB)': 42.0,    # ~336 Gbps effective
    'NVLink (intra-node)': 450.0 # NVLink 4.0 bidirectional
}

def transfer_time_ms(size_bytes, bw_gbps):
    return (size_bytes / (bw_gbps * 1e9)) * 1000

seq_test = [1024, 4096, 16384, 32768]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(seq_test))
width = 0.2

for i, (name, bw) in enumerate(INTERCONNECTS.items()):
    times = [transfer_time_ms(kv_cache_size_bytes(s), bw) for s in seq_test]
    ax.bar(x + i * width, times, width, label=name)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels([f'{s//1024}K' for s in seq_test])
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Transfer Time (ms)')
ax.set_title('KV Cache Transfer Latency by Interconnect')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 3. Prefill/Decode Pool Sizing

Optimal pool ratio depends on workload characteristics:
- **Long prompts, short outputs** → more prefill GPUs
- **Short prompts, long outputs** → more decode GPUs

We model throughput as a function of pool split ratio.

In [ ]:
TOTAL_GPUS = 16
PREFILL_TFLOPS = 312  # A100 peak FP16
DECODE_MEM_BW = 2000  # GB/s A100 HBM

def prefill_throughput(n_prefill_gpus, avg_prompt_tokens=2048):
    """Tokens/s the prefill pool can process (compute-bound)."""
    flops_per_token = 2 * 7e9  # ~7B param model, 2 FLOPs/param
    tokens_per_gpu_per_s = (PREFILL_TFLOPS * 1e12) / flops_per_token
    return n_prefill_gpus * tokens_per_gpu_per_s

def decode_throughput(n_decode_gpus, avg_output_tokens=256):
    """Requests/s the decode pool can serve (memory-bound)."""
    bytes_per_token = 2 * 7e9 * 2 / n_decode_gpus  # model weights read per token, TP
    tokens_per_s = (DECODE_MEM_BW * 1e9 * n_decode_gpus) / (2 * 7e9 * 2)
    return tokens_per_s / avg_output_tokens  # requests/s

ratios = np.arange(1, TOTAL_GPUS)
prefill_tp = [prefill_throughput(r) for r in ratios]
decode_tp = [decode_throughput(TOTAL_GPUS - r) for r in ratios]
# System throughput limited by bottleneck
system_tp = [min(p / 2048, d) for p, d in zip(prefill_tp, decode_tp)]  # normalize to req/s

plt.figure(figsize=(8, 4))
plt.plot(ratios, system_tp, 'o-', color='#2563eb', linewidth=2)
best = np.argmax(system_tp)
plt.axvline(ratios[best], color='red', linestyle='--', label=f'Optimal: {ratios[best]} prefill GPUs')
plt.xlabel('Prefill GPUs (out of 16 total)')
plt.ylabel('System Throughput (req/s)')
plt.title('Pool Sizing: System Throughput vs Prefill/Decode Split')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Optimal split: {ratios[best]} prefill + {TOTAL_GPUS - ratios[best]} decode GPUs")

## 4. Mooncake: Prediction-Based Disaggregated Scheduling

**Mooncake** (Moonshot AI, 2024) introduces:
- **Output-length prediction** to pre-allocate decode slots
- **Conductor** scheduler that routes requests to prefill/decode pools
- **KV cache pipelining**: overlap transfer with ongoing decode batches
- **Prefix caching** across the prefill pool for shared system prompts

Key insight: if you can predict output length, you can schedule decode resources
before prefill finishes, hiding transfer latency.

In [ ]:
# Simulate Mooncake's pipelined KV transfer vs naive sequential
def simulate_mooncake(n_requests=100, avg_prefill_ms=50, avg_decode_ms=200,
                      transfer_ms=5, prediction_accuracy=0.85):
    """Compare naive (wait for transfer) vs Mooncake (pipelined transfer)."""
    np.random.seed(42)
    prefill_times = np.random.exponential(avg_prefill_ms, n_requests)
    decode_times = np.random.exponential(avg_decode_ms, n_requests)
    
    # Naive: prefill -> transfer -> decode (sequential)
    naive_latencies = prefill_times + transfer_ms + decode_times
    
    # Mooncake: overlap transfer with decode batch formation
    # When prediction is correct, transfer is hidden; when wrong, add reallocation penalty
    correct = np.random.random(n_requests) < prediction_accuracy
    mooncake_latencies = prefill_times + decode_times + np.where(correct, 0, transfer_ms * 2)
    
    return naive_latencies, mooncake_latencies

naive, mooncake = simulate_mooncake()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(naive, bins=30, alpha=0.7, label='Naive Sequential', color='#ef4444')
axes[0].hist(mooncake, bins=30, alpha=0.7, label='Mooncake Pipelined', color='#2563eb')
axes[0].set_xlabel('End-to-End Latency (ms)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Latency Distribution')

percentiles = [50, 90, 95, 99]
naive_p = np.percentile(naive, percentiles)
moon_p = np.percentile(mooncake, percentiles)
x = np.arange(len(percentiles))
axes[1].bar(x - 0.15, naive_p, 0.3, label='Naive', color='#ef4444')
axes[1].bar(x + 0.15, moon_p, 0.3, label='Mooncake', color='#2563eb')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'P{p}' for p in percentiles])
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Tail Latency Comparison')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"Median improvement: {np.median(naive) - np.median(mooncake):.1f} ms ({(1 - np.median(mooncake)/np.median(naive))*100:.1f}%)")

## 5. Throughput: Aggregated vs Disaggregated Serving

We compare total system throughput under varying request rates for:
- **Aggregated**: all GPUs handle both prefill and decode (standard vLLM)
- **Disaggregated**: separate pools with KV transfer overhead

In [ ]:
def simulate_throughput(arrival_rates, total_gpus=16, prefill_ratio=0.375):
    """Simulate throughput for aggregated vs disaggregated under load."""
    n_prefill = int(total_gpus * prefill_ratio)
    n_decode = total_gpus - n_prefill
    
    # Aggregated: each GPU does both, interference between phases
    # Prefill preempts decode -> decode bubbles under high load
    agg_max_throughput = total_gpus * 8  # req/s baseline
    
    # Disaggregated: no interference, but transfer overhead
    disagg_prefill_cap = n_prefill * 20  # prefill is faster per-request
    disagg_decode_cap = n_decode * 10
    disagg_max = min(disagg_prefill_cap, disagg_decode_cap)
    
    agg_results = []
    disagg_results = []
    
    for rate in arrival_rates:
        # Aggregated: throughput degrades due to prefill/decode interference
        utilization = rate / agg_max_throughput
        interference = 1.0 - 0.3 * min(utilization, 1.0) ** 2  # quadratic degradation
        agg_tp = min(rate, agg_max_throughput * interference)
        agg_results.append(agg_tp)
        
        # Disaggregated: linear until capacity, small transfer penalty
        transfer_penalty = 0.95  # 5% overhead from KV transfer
        disagg_tp = min(rate, disagg_max * transfer_penalty)
        disagg_results.append(disagg_tp)
    
    return agg_results, disagg_results

rates = np.linspace(10, 200, 50)
agg_tp, disagg_tp = simulate_throughput(rates)

plt.figure(figsize=(8, 5))
plt.plot(rates, agg_tp, 'o-', label='Aggregated (vLLM-style)', color='#ef4444', markersize=3)
plt.plot(rates, disagg_tp, 's-', label='Disaggregated (Mooncake-style)', color='#2563eb', markersize=3)
plt.plot(rates, rates, '--', color='gray', alpha=0.5, label='Ideal (rate = throughput)')
plt.xlabel('Arrival Rate (req/s)')
plt.ylabel('Achieved Throughput (req/s)')
plt.title('Aggregated vs Disaggregated: Throughput Under Load')
plt.legend()
plt.tight_layout()
plt.show()

crossover = next(i for i in range(len(rates)) if disagg_tp[i] > agg_tp[i])
print(f"Disaggregated wins above ~{rates[crossover]:.0f} req/s arrival rate")
print(f"At 150 req/s: Aggregated={agg_tp[37]:.1f}, Disaggregated={disagg_tp[37]:.1f} "
      f"(+{(disagg_tp[37]/agg_tp[37]-1)*100:.0f}%)")

In [ ]:
# 6. Sensitivity: how much does KV transfer overhead eat into disaggregated advantage?
transfer_overheads = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]  # fraction of total latency
seq_lengths = [1024, 4096, 16384]

fig, ax = plt.subplots(figsize=(8, 4))
for seq in seq_lengths:
    # Speedup = (prefill+decode) / (prefill+transfer+decode) * pool_efficiency
    base_prefill_ms = seq * 0.02  # ~20us per token prefill
    base_decode_ms = 200  # fixed decode time
    total_base = base_prefill_ms + base_decode_ms
    
    speedups = []
    for overhead in transfer_overheads:
        transfer_ms = total_base * overhead
        # Disaggregated benefit: 1.4x from specialization, minus transfer cost
        disagg_time = (base_prefill_ms / 1.4 + transfer_ms + base_decode_ms / 1.3)
        speedups.append(total_base / disagg_time)
    
    ax.plot(np.array(transfer_overheads) * 100, speedups, 'o-', label=f'seq={seq}')

ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Break-even')
ax.set_xlabel('Transfer Overhead (% of total latency)')
ax.set_ylabel('Speedup vs Aggregated')
ax.set_title('Disaggregated Speedup Sensitivity to Transfer Cost')
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

1. **KV cache transfer is the critical cost** — RDMA (400Gb) keeps it under 2ms for 4K sequences
2. **Pool sizing** depends on prompt/output ratio — typical sweet spot is 30-40% prefill GPUs
3. **Mooncake's pipelining** hides transfer latency via output-length prediction (85%+ accuracy)
4. **Disaggregated wins at scale** — crossover point around 60-80% utilization where prefill/decode interference dominates in aggregated systems
5. **Transfer overhead > 15%** erases the disaggregated advantage — fast interconnect is mandatory